In [27]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import gc


np.random.seed(31337)
print("Библиотеки загружены!")

Библиотеки загружены!


In [28]:

data = pd.read_csv("data/contest_1/raw_data/train.csv")


grouped_data = data.groupby(['user', 'track']).agg(
    total_time=('time', 'sum'),
    count=('user', 'size')
).reset_index()


grouped_data = grouped_data[grouped_data["total_time"] > 0.5].copy().reset_index(drop=True)


grouped_data["if_test"] = np.random.random(len(grouped_data)) > 0.8

print(f"Всего уникальных пар после фильтрации: {len(grouped_data)}")

Всего уникальных пар после фильтрации: 891261


In [29]:

user_counts = grouped_data.loc[~grouped_data["if_test"], "user"].value_counts()
users = set(user_counts[user_counts >= 5].index.values)


track_counts = grouped_data.loc[~grouped_data["if_test"], "track"].value_counts()
tracks = set(track_counts[track_counts >= 3].index.values)


train_data = grouped_data[~grouped_data["if_test"] & grouped_data["user"].isin(users) & grouped_data["track"].isin(tracks)]
test_data = grouped_data[grouped_data["if_test"] & grouped_data["user"].isin(users) & grouped_data["track"].isin(tracks)]

print(f"Train: {len(train_data)}, Test: {len(test_data)}")
print(f"Уникальных юзеров: {len(users)}, Уникальных треков: {len(tracks)}")

Train: 713418, Test: 177843
Уникальных юзеров: 10000, Уникальных треков: 16198


In [33]:

user_to_idx = {u: i for i, u in enumerate(users)}
track_to_idx = {t: i for i, t in enumerate(tracks)}


def create_sparse_matrix(df):
    rows = df['user'].map(user_to_idx)
    cols = df['track'].map(track_to_idx)
    
    valid_mask = rows.notna() & cols.notna()
    rows = rows[valid_mask].astype(int)
    cols = cols[valid_mask].astype(int)
    
    data = np.ones(len(rows)) 
    shape = (len(users), len(tracks)) 
    return sp.csr_matrix((data, (rows, cols)), shape=shape)


train_matrix = create_sparse_matrix(train_data)
test_matrix = create_sparse_matrix(test_data)


print(f"Размер train_matrix: {train_matrix.shape}") 


Размер train_matrix: (10000, 16198)


In [34]:
def fit_ease(X, lambda_=15000.0):
    """
    X: разреженная матрица (users x items)
    lambda_: параметр регуляризации
    """
    print("Вычисляем матрицу Грама (X^T * X)...")
    
    G = (X.T @ X).toarray()
    
    print("Добавляем регуляризацию и считаем обратную матрицу...")
   
    G += np.eye(G.shape[0]) * lambda_
    
    
    P = np.linalg.inv(G)
    
    print("Формируем матрицу весов B...")
    
    B = P / (-np.diag(P))
    np.fill_diagonal(B, 0.0)
    
    return B


lambda_val = 15000.0
B_matrix = fit_ease(train_matrix, lambda_=lambda_val)
print("EASE обучен!")

Вычисляем матрицу Грама (X^T * X)...
Добавляем регуляризацию и считаем обратную матрицу...
Формируем матрицу весов B...
EASE обучен!


In [35]:

submit_data = pd.read_csv("data/contest_1/raw_data/test.csv")


filtered_submit_data = submit_data[
    submit_data["user"].isin(users) & 
    submit_data["track"].isin(tracks)
].drop_duplicates().copy()

print(f"Строк для предсказания: {len(filtered_submit_data)}")


users_idx = filtered_submit_data['user'].map(user_to_idx).astype(int).values
tracks_idx = filtered_submit_data['track'].map(track_to_idx).astype(int).values


print("Считаем предсказания для всех юзеров...")
all_scores = train_matrix @ B_matrix 


all_scores_dense = np.asarray(all_scores) 
scores = all_scores_dense[users_idx, tracks_idx]


filtered_submit_data["score"] = scores

Строк для предсказания: 647307
Считаем предсказания для всех юзеров...


In [36]:

scored_submit_data = pd.merge(
    submit_data,
    filtered_submit_data,
    left_on=["user", "track"],
    right_on=["user", "track"],
    how="left"
)

scored_submit_data["score"] = scored_submit_data["score"].map(
    lambda s: np.random.random() * 0.01 if pd.isna(s) else np.exp(s)
)


print(scored_submit_data.head())


version = 7  
file_path = f"data/contest_1/processed/submit_v{version}.csv"
scored_submit_data.to_csv(file_path, index=False)
print(f"Файл сохранен: {file_path}")

                               user  track     score
0  abb1b98c5cbcff30d8ca270ca437ddb8  10901  1.000268
1  bae0a9b4027abefe8f55351183f19604   7685  1.005050
2  e981fcd33e48ea5de3e7e2f784a47566   9346  1.001481
3  41806b51385ff04e6edcd28096724778    490  1.004284
4  84d430b459ad7fbfe20a8b3902848a82   1027  1.004585
Файл сохранен: data/contest_1/processed/submit_v7.csv
